# 项目组织与工程工具

学习目标：能解释包解析与依赖锁定，运行静态检查、格式检查和本地持续集成入口。

前置知识：Node.js 文件运行、ES 模块、CommonJS、测试及 JSON 配置。

适用版本：Node.js 24.11.0、npm 11.6.1、ESLint 10.10.0、Prettier 3.6.2；语言示例以 ECMAScript 2025 为基线。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/25-project-tools/。

1. [manifest.mjs](scripts/25-project-tools/manifest.mjs)：读取实际环境清单。
2. [package.json](scripts/25-project-tools/fixture/package.json)：明确 ESM 模式与公开接口。
3. [number.js](scripts/25-project-tools/fixture/number.js)：包内部函数。
4. [index.js](scripts/25-project-tools/fixture/index.js)：从内部映射导出公开函数。
5. [legacy.cjs](scripts/25-project-tools/fixture/legacy.cjs)：同包内明确的 CommonJS 文件。
6. [consumer.mjs](scripts/25-project-tools/fixture/consumer.mjs)：实际解析和未导出子路径拒绝。
7. [eslint.config.mjs](scripts/25-project-tools/eslint.config.mjs)：最小真实可执行的 ESLint 平面配置。
8. [clean.mjs](scripts/25-project-tools/clean.mjs)：同时通过 lint 和格式检查的输入。
9. [lint-failure.mjs](scripts/25-project-tools/lint-failure.mjs)：只用于静态检查的独立反例。
10. [ci.mjs](scripts/25-project-tools/ci.mjs)：实际运行的连续检查。
11. [compatibility.mjs](scripts/25-project-tools/compatibility.mjs)：限定范围的兼容函数及动态作用域差异。

Step 1：读取清单并核对锁文件。

```bash
node scripts/25-project-tools/manifest.mjs
```

Step 2：运行本章连续检查。

```bash
npm run check:25
```

Step 3：运行兼容示例。

```bash
node scripts/25-project-tools/compatibility.mjs
```

Step 4：单独运行预期失败的静态示例。

```bash
node node_modules/eslint/bin/eslint.js --config scripts/25-project-tools/eslint.config.mjs scripts/25-project-tools/lint-failure.mjs
```

## 1 运行时、包管理器与依赖清单

Node.js 执行 JavaScript 并提供文件、进程等宿主 API；npm 安装与解析依赖、运行包脚本、生成分发包。安装 npm 包不会自动改变 JavaScript 语法，也不会把 Node.js 的文件 API 加入浏览器。

package.json 描述包名、版本、入口、脚本与直接依赖。dependencies 是消费者运行所需的包，devDependencies 是开发、测试或格式化等工具；本课程语言示例没有运行依赖，ESLint 与 Prettier 放在开发依赖中。private 为 true 阻止意外发布。engines 声明兼容条件，通常是提示约束，不会替你下载或切换 Node.js。

本目录的 package.json 固定工具版本，package-lock.json 锁定解析出的完整依赖树和完整性信息。下面脚本读取实际配置；它说明根包与子包可以有不同职责。

配套 [manifest.mjs](scripts/25-project-tools/manifest.mjs)：

```javascript
import assert from "node:assert/strict";
import { readFile } from "node:fs/promises";
const packageJson = JSON.parse(await readFile("package.json", "utf8"));
const lock = JSON.parse(await readFile("package-lock.json", "utf8"));
assert.equal(packageJson.private, true);
assert.equal(packageJson.devDependencies.eslint, "10.10.0");
assert.equal(packageJson.devDependencies.prettier, "3.6.2");
assert.deepEqual(lock.packages[""].devDependencies, packageJson.devDependencies);
console.log("private tools locked", lock.lockfileVersion); // → private tools locked 3
```

## 2 锁文件、npm ci 与脚本入口

npm install 用于增加或调整依赖，可能更新锁文件；团队应同时保存 package.json 与锁文件。npm ci 要求已有锁文件，依赖声明不匹配时失败，而不是修改锁；它会先移除该项目已有 node_modules，再按锁文件安装。因此不要在依赖被其他进程使用时执行。

npm run 会把本地 node_modules/.bin 加入脚本 PATH，并以包根目录作为工作目录，使团队通过同一命令调用本地工具。本章的 lint:25、format:25 与 check:25 都来自实际根 package.json。下面两步分别完成可复现安装与检查；ignore-scripts 禁止依赖安装钩子，本章工具不需要这些钩子。no-audit 只关闭安装时的远程审计，不代表依赖已经安全。

Step 1：按锁文件安装本地工具。

```bash
npm ci --ignore-scripts --no-audit --no-fund
```

Step 2：运行本章连续检查入口。

```bash
npm run check:25
```

## 3 模块识别、包入口与内部导入

Node.js 中 .mjs 明确为 ESM，.cjs 明确为 CommonJS；.js 由最近的 package.json 的 type 等规则判定。显式写 type 能避免依赖语法检测。CommonJS 使用 require 和 module.exports，ESM 使用 import/export；两者互操作需要考虑导出形状、异步模块与解析规则。

exports 声明公开包入口及允许的子路径；它优先于 main，并能限制按包名导入未公开子路径。这个封装边界不等于文件系统权限。imports 的 # 前缀名称只在所属包内部生效；它能表达内部路径映射。下面子包用自身包名验证 exports，用 #number 验证 imports。

配套 [fixture/package.json](scripts/25-project-tools/fixture/package.json)：

```json
{
  "name": "notebook-tools-fixture",
  "version": "1.0.0",
  "private": true,
  "type": "module",
  "exports": { ".": "./index.js" },
  "imports": { "#number": "./number.js" }
}
```

配套 [fixture/number.js](scripts/25-project-tools/fixture/number.js)：

```javascript
export const triple = (value) => value * 3;
```

配套 [fixture/index.js](scripts/25-project-tools/fixture/index.js)：

```javascript
import { triple } from "#number";
export { triple };
```

配套 [fixture/legacy.cjs](scripts/25-project-tools/fixture/legacy.cjs)：

```javascript
module.exports = { label: "CommonJS" };
```

配套 [fixture/consumer.mjs](scripts/25-project-tools/fixture/consumer.mjs)：

```javascript
import assert from "node:assert/strict";
import { triple } from "notebook-tools-fixture";
import legacy from "./legacy.cjs";
assert.equal(triple(4), 12);
assert.equal(legacy.label, "CommonJS");
await assert.rejects(import("notebook-tools-fixture/number.js"), {
  code: "ERR_PACKAGE_PATH_NOT_EXPORTED",
});
console.log("exports imports CommonJS checked"); // → exports imports CommonJS checked
```

## 4 ESLint 的规则与格式化职责

ESLint 按语法与配置规则报告问题；本例启用 no-undef、no-unused-vars 和 eqeqeq，分别检查未声明名称、未使用变量和宽松相等。languageOptions 指定语法版本及宿主全局名称，不会把代码转换成旧语法，也不会提供相应运行时 API。

平面配置（flat config）导出配置对象数组。这里显式传 --config，确保只检查指定示例；故意出错的 lint-failure.mjs 独立运行。规则失败返回非零状态，适合被持续集成捕获。

Prettier 负责一致的排版，--check 检查现有格式，--write 修改文件。它不检查业务逻辑。这里没有 ESLint 排版规则，因此不额外引入处理排版冲突的配置包；格式化范围也只覆盖 clean.mjs。

配套 [eslint.config.mjs](scripts/25-project-tools/eslint.config.mjs)：

```javascript
export default [
  {
    files: ["**/*.mjs"],
    languageOptions: {
      ecmaVersion: 2025,
      sourceType: "module",
      globals: { console: "readonly" },
    },
    rules: {
      "no-undef": "error",
      "no-unused-vars": "error",
      eqeqeq: ["error", "always"],
    },
  },
];
```

配套 [clean.mjs](scripts/25-project-tools/clean.mjs)：

```javascript
export function triple(value) {
  return value * 3;
}
console.log(triple(4)); // → 12
```

配套 [lint-failure.mjs](scripts/25-project-tools/lint-failure.mjs)：

```javascript
console.log(notDeclared == 1);
// → ESLint 返回 1，分别包含 no-undef 和 eqeqeq；不作为正常脚本运行。
```

## 5 把检查放进持续集成

持续集成（continuous integration，CI）是在每次集成代码时自动执行约定检查。本例提供可在本地或 CI 服务调用的同一入口：顺序执行 lint、格式检查、模块消费者与实际函数。任一步失败就停止，保留该步骤诊断。

CI 主机先安装声明的 Node.js/npm 版本，再运行 npm ci 和 npm run check:25。这里只提供可复用入口，不在仓库根目录开启远程工作流。execFileSync 使用参数数组启动本机 Node.js，不把路径拼成 shell 命令；它属于检查工具，业务异步代码不应因此改成阻塞 I/O。

配套 [ci.mjs](scripts/25-project-tools/ci.mjs)：

```javascript
import { execFileSync } from "node:child_process";
const checks = [
  ["node_modules/eslint/bin/eslint.js", "--config", "scripts/25-project-tools/eslint.config.mjs", "scripts/25-project-tools/clean.mjs"],
  ["node_modules/prettier/bin/prettier.cjs", "--check", "scripts/25-project-tools/clean.mjs"],
  ["scripts/25-project-tools/fixture/consumer.mjs"],
  ["scripts/25-project-tools/clean.mjs"],
];
for (const args of checks) {
  execFileSync(process.execPath, args, { stdio: "inherit" });
}
console.log("CI checks passed"); // → CI checks passed；任一步失败则不会到达这里
```

## 6 兼容性、依赖安全与动态代码边界

目标环境包含语法解析能力、内置 API、宿主 API 和包版本，必须分别核对。polyfill 在运行时补齐可实现的 API；语法转换在运行前把新语法改写为目标环境能够解析的形式。只补一个方法，不能让旧解析器识别 await；只转换箭头函数，也不会自动补出 Promise。Babel 的转换与 polyfill 配置因此是不同环节。

下面的 sortedCopy 是本例限定为普通数值数组的兼容函数，并非完整 toSorted polyfill。完整补丁还需要遵守稀疏数组、属性访问与比较函数等标准行为，不能因为一个测试通过就宣称完全兼容。

依赖安全需要审查来源、版本变更和安装脚本。npm audit 查询已知漏洞信息，不能证明没有未知问题；审计依赖外部服务，应单独安排，不让确定性语言示例依赖网络。不要把 audit fix --force 当作无需评审的修复，它可能改变兼容性。

eval 和 Function 都能把字符串作为代码；直接 eval 可访问当前词法作用域，Function 构造的函数在全局环境中执行，不捕获创建位置的局部变量。不要把用户输入拼进去；数据解析用 JSON.parse，可选行为用函数映射。

配套 [compatibility.mjs](scripts/25-project-tools/compatibility.mjs)：

```javascript
import assert from "node:assert/strict";
function sortedCopy(values) {
  return [...values].sort((left, right) => left - right);
}
const input = [3, 1, 2];
assert.deepEqual(sortedCopy(input), input.toSorted((left, right) => left - right));
assert.deepEqual(input, [3, 1, 2]);
const localFactor = 3;
console.log(eval("localFactor * 2")); // → 6；这里只执行自己固定写出的代码
console.log(Function("return typeof localFactor")()); // → undefined
const operations = { double: (value) => value * 2 };
assert.equal(operations.double(5), 10);
console.log("compatibility checked"); // → compatibility checked
```

## 本章小结

- 包清单表达直接依赖和入口，锁文件记录完整解析结果，npm ci 重现安装。
- 模块模式、公开导出和内部映射分别影响文件如何运行、消费者能导入什么。
- 静态检查、格式检查、实际测试和安全审计提供不同证据。

## 练习

1. 给 fixture 的 exports 增加明确的 ./number 子路径并更新消费者；标准：该子路径导入成功，其余未声明路径仍拒绝。
2. 将 clean.mjs 中一个名称拼错，运行 lint:25；标准：no-undef 导致非零退出，恢复后 check:25 通过。
3. 给 sortedCopy 增加负数和重复值输入；标准：排序正确且不修改原数组，同时说明这些测试为什么还不能证明完整 polyfill 兼容。

## 参考与引用来源

- Node.js 24.11.0：[Packages](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html)，Determining module system、Package entry points、Subpath imports、Self-referencing；[ESM 互操作](https://nodejs.org/download/release/v24.11.0/docs/api/esm.html#interoperability-with-commonjs)、[child_process.execFileSync](https://nodejs.org/download/release/v24.11.0/docs/api/child_process.html#child_processexecfilesyncfile-args-options)。
- npm：[package.json](https://docs.npmjs.com/cli/v11/configuring-npm/package-json/)、[锁文件](https://docs.npmjs.com/cli/v11/configuring-npm/package-lock-json/)、[npm ci](https://docs.npmjs.com/cli/v11/commands/npm-ci/)、[npm scripts](https://docs.npmjs.com/cli/v11/using-npm/scripts/)、[npm audit](https://docs.npmjs.com/cli/v11/commands/npm-audit/)：本文使用 npm 11.6.1 已具备的参数，v11 在线文档可能包含后续小版本选项。
- ESLint：[平面配置](https://eslint.org/docs/latest/use/configure/configuration-files)、[入门与运行条件](https://eslint.org/docs/latest/use/getting-started)：规则数组和本地工具入口。
- raw.githubusercontent.com：[ESLint 10.10.0 package.json](https://raw.githubusercontent.com/eslint/eslint/v10.10.0/package.json)、[Prettier 3.6.2 package.json](https://raw.githubusercontent.com/prettier/prettier/3.6.2/package.json)：固定版本的 engines 声明。
- Prettier：[安装与 --check/--write](https://prettier.io/docs/install)：本地固定版本与排版检查。
- Babel：[Usage Guide](https://babeljs.io/docs/usage)：转换语法、目标环境和 polyfill 的分工。
- TC39（ECMA-262 第 16 版）：[§19.2.1 eval](https://tc39.es/ecma262/2025/multipage/global-object.html#sec-eval-x)、[§20.2.1 Function](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-function-constructor)：动态代码环境。